### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp03'
DATA_URLS = {'city_assets.csv': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp03/city_assets.csv'}
SCHEMA = {'city_assets.csv': {'kind': 'csv', 'fields': ['asset_id', 'asset_type', 'condition', 'longitude', 'latitude']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP03 — CSV to GIS Dataset

CSV → DataFrame → GeoDataFrame → CRS → QA → Export

Στόχος: να αποδείξεις πότε ένας πίνακας έγινε πραγματικά GIS dataset. Έλεγξε schema, longitude/latitude, geometry, CRS και export πριν προχωρήσεις.

## 1. CSV schema και ανάγνωση

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
df = pd.read_csv(DATA_ROOT / "gp03/city_assets.csv")
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isna().sum())


## 2. Συντεταγμένες και geometry

Χρησιμοποίησε longitude ως X και latitude ως Y.

In [ ]:
# gdf = gpd.GeoDataFrame(..., geometry=gpd.points_from_xy(...), crs="EPSG:4326")
# Εμφάνισε geometry types και bounds


## 3. CRS: set_crs και to_crs

In [ ]:
# Έλεγξε το CRS του gdf.
# Δημιούργησε projected = gdf.to_crs("EPSG:2100") μόνο για σύγκριση.


## 4. GeoPython QA

In [ ]:
# Έλεγξε rows, CRS, missing attributes, Point geometry, valid geometry, μοναδικά IDs και bounds.


## 5. Export και re-read

In [ ]:
output_dir = Path("outputs/gp03")
output_dir.mkdir(parents=True, exist_ok=True)
# Κάνε export σε GeoJSON και GeoPackage.
# Διάβασε ξανά ένα output με gpd.read_file().


## Mini Challenge

Εξήγαγε μόνο τα assets με condition == 'good'.

In [ ]:
# good = ...
# good.to_file(...)
